# NewMenna — Kaggle Version
**Spaced Egyptian characters → assembled words using ByT5-small**
#
### What's fixed vs original:
- Kaggle paths (`/kaggle/input/`, `/kaggle/working/`)
- Session crash fix: `del trainer` + `empty_cache()` before final eval
- `MAX_INPUT_LEN` / `MAX_TARGET_LEN` reduced for ByT5 byte-level reality
- `FINAL_EVAL_LIMIT` capped — beam search never OOMs
- `dataloader_num_workers=2` (Kaggle supports this, unlike Colab)
- `NUM_TRAIN_EPOCHS=10` + `EARLY_STOPPING_PATIENCE=4` to reach 80s EM
- `label_smoothing=0.1` for better generalization
- `generation_num_beams=2` during training eval for accurate EM tracking

## Cell 1 — Install missing packages
(Kaggle has transformers/torch pre-installed; only missing ones needed)

In [1]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

pip_install("python-Levenshtein")
pip_install("evaluate")
# transformers, datasets, accelerate, sentencepiece already on Kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


## Cell 2 — Imports & config

In [2]:
import gc
import os
import re
import math
import random
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import Levenshtein

from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import (
    ByT5Tokenizer,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
    set_seed,
)

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ── Seeds ─────────────────────────────────────────────────────────────────────
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ── Paths — update DATASET_SLUG to match your Kaggle dataset name ─────────────
# Example: if you uploaded as "newmenna-data", slug = "newmenna-data"
DATASET_SLUG = "your-dataset-slug"   # ← CHANGE THIS

CANDIDATE_PATHS = [
    Path(f"/kaggle/input/datasets/mennamostafan/cjarcters/character_level.csv"),
    Path("/kaggle/input/character_level.csv"),
    Path("/kaggle/working/character_level.csv"),
    Path("./character_level.csv"),
]
CSV_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        "character_level.csv not found. "
        "Upload your dataset to Kaggle and set DATASET_SLUG above."
    )

# All outputs go to /kaggle/working/ — this persists after the session ends
OUTPUT_DIR = Path("/kaggle/working/newmenna_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "google/byt5-small"

# ByT5 is BYTE-LEVEL: each character ≈ 1–4 tokens.
# Original 192/160 was wasteful and caused OOM. 128/96 covers ~95%+ of data.
MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 96

# ── Training ──────────────────────────────────────────────────────────────────
NUM_TRAIN_EPOCHS        = 10    # was 3  → enough epochs to reach 80s EM
EARLY_STOPPING_PATIENCE = 4     # was 2  → don't quit too early
LEARNING_RATE           = 3e-4
TRAIN_BATCH_SIZE        = 8
EVAL_BATCH_SIZE         = 8     # was 16 → halved to avoid OOM during generate()
GRAD_ACCUM              = 4     # effective batch = 32, more stable gradients
EVAL_STEPS              = 400
EVAL_SUBSET_SIZE        = 500
LABEL_SMOOTHING         = 0.1   # was 0.0 → helps generalization

# ── Inference ─────────────────────────────────────────────────────────────────
FINAL_NUM_BEAMS  = 4
FINAL_EVAL_LIMIT = 1000         # was None → cap to avoid post-training OOM

print("CSV_PATH   :", CSV_PATH)
print("OUTPUT_DIR :", OUTPUT_DIR)
print("MODEL_NAME :", MODEL_NAME)
print("CUDA       :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0))
    print("VRAM GB    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

CSV_PATH   : /kaggle/input/datasets/mennamostafan/cjarcters/character_level.csv
OUTPUT_DIR : /kaggle/working/newmenna_outputs
MODEL_NAME : google/byt5-small
CUDA       : True
GPU        : Tesla T4
VRAM GB    : 15.64


## Cell 3 — Normalization

In [3]:
MDC_TO_EGYPTO = {
    "A": "ꜣ", "i": "ꞽ", "a": "ꜥ",
    "H": "ḥ", "x": "ḫ", "X": "ẖ",
    "S": "š", "T": "ṯ", "D": "ḏ",
}
EGYPTO_SIGNATURES = set("ꜣꞽꜥḥḫẖšṯḏ")
APOSTROPHES = {"ʾ": "'", "\u2018": "'", "\u2019": "'"}
CANONICAL_REPLACEMENTS = {
    "ẖ": "ẖ",
    "H̱": "Ḥ",
    "tʾ": "t'",
    "rʾ": "r'",
}


def clean_text(s: str) -> str:
    s = unicodedata.normalize("NFC", str(s))
    s = s.replace("\u00a0", " ")   # non-breaking space → regular space
    return re.sub(r"\s+", " ", s).strip()


def looks_like_mdc(s: str) -> bool:
    if any(c in EGYPTO_SIGNATURES for c in s):
        return False
    return bool(re.search(r"[AHSTDXxia]", s))


def canonicalize(s: str) -> str:
    s = unicodedata.normalize("NFC", str(s))
    for a, b in CANONICAL_REPLACEMENTS.items():
        s = s.replace(a, b)
    return s


def normalize_input(s: str) -> str:
    s = clean_text(s)
    for k, v in APOSTROPHES.items():
        s = s.replace(k, v)
    if looks_like_mdc(s):
        s = "".join(MDC_TO_EGYPTO.get(c, c) for c in s)
    s = canonicalize(s)
    s = re.sub(r"[()<>\[\],;|\-]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def normalize_target(s: str) -> str:
    s = clean_text(s)
    s = canonicalize(s)
    return re.sub(r"\s+", " ", s).strip()


def strip_spaces(s: str) -> str:
    return s.replace(" ", "")


# Quick sanity checks
assert normalize_input("ḥ t p ẖ n m w") == "ḥ t p ẖ n m w"
assert normalize_target("ḥtp ẖnmw") == "ḥtp ẖnmw"
print("Normalization tests passed ✓")

Normalization tests passed ✓


## Cell 4 — Load CSV & split

In [4]:
df = pd.read_csv(CSV_PATH)
df = df.rename(columns={"characters": "input_text", "clean_text": "target_text"})
df = df[["input_text", "target_text"]].dropna().copy()

df["input_text"]  = df["input_text"].astype(str).map(normalize_input)
df["target_text"] = df["target_text"].astype(str).map(normalize_target)
df = df[(df["input_text"] != "") & (df["target_text"] != "")].reset_index(drop=True)

# Length filter
before = len(df)
df = df[
    (df["input_text"].str.len()  <= MAX_INPUT_LEN) &
    (df["target_text"].str.len() <= MAX_TARGET_LEN)
].reset_index(drop=True)
print(f"After length filter      : {len(df):,}  (dropped {before - len(df):,})")

# Assembly integrity: input without spaces must equal target without spaces
before = len(df)
clean_mask = df["input_text"].map(strip_spaces) == df["target_text"].map(strip_spaces)
df = df[clean_mask].reset_index(drop=True)
print(f"After assembly filter    : {len(df):,}  (dropped {before - len(df):,})")

# Deduplicate exact pairs
before = len(df)
df = df.drop_duplicates(subset=["input_text", "target_text"]).reset_index(drop=True)
print(f"After exact-pair dedup   : {len(df):,}  (dropped {before - len(df):,})")

# Tag ambiguous inputs (same input, different targets)
amb_counts      = df.groupby("input_text")["target_text"].nunique()
ambiguous_inputs = set(amb_counts[amb_counts > 1].index)
df["_ambiguous"] = df["input_text"].isin(ambiguous_inputs)
print(f"Ambiguous inputs         : {len(ambiguous_inputs):,}")


def length_bucket(s: str) -> str:
    n = len(s.split())
    if n <= 3:  return "short"
    if n <= 8:  return "medium"
    if n <= 20: return "long"
    return "xlong"


df["_bucket"] = df["target_text"].map(length_bucket)


def split_no_leakage(frame, test_size=0.10, val_size=0.10, seed=42):
    unamb = frame[~frame["_ambiguous"]].reset_index(drop=True)
    amb   = frame[ frame["_ambiguous"]].reset_index(drop=True)

    trainval, test = train_test_split(
        unamb, test_size=test_size, random_state=seed, stratify=unamb["_bucket"],
    )
    train, val = train_test_split(
        trainval,
        test_size=val_size / (1 - test_size),
        random_state=seed,
        stratify=trainval["_bucket"],
    )
    # Ambiguous examples only in train (safe — can't leak what's never in val/test)
    train = pd.concat([train, amb], ignore_index=True)
    train_inputs = set(train["input_text"])
    val  = val[ ~val["input_text"].isin(train_inputs)].reset_index(drop=True)
    test = test[~test["input_text"].isin(train_inputs)].reset_index(drop=True)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


train_df, val_df, test_df = split_no_leakage(df, seed=SEED)
print(f"\ntrain={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[["input_text", "target_text"]], preserve_index=False),
    "val":   Dataset.from_pandas(val_df[  ["input_text", "target_text"]], preserve_index=False),
    "test":  Dataset.from_pandas(test_df[ ["input_text", "target_text"]], preserve_index=False),
})
print(dataset)

After length filter      : 108,019  (dropped 5,437)
After assembly filter    : 104,363  (dropped 3,656)
After exact-pair dedup   : 81,364  (dropped 22,999)
Ambiguous inputs         : 76

train=65,120  val=8,122  test=8,122
DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 65120
    })
    val: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 8122
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 8122
    })
})


## Cell 5 — Tokenizer & model

In [5]:
tokenizer = ByT5Tokenizer.from_pretrained(MODEL_NAME)
model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model.gradient_checkpointing_enable()
model.config.use_cache = False

print(f"Model parameters : {model.num_parameters():,}")
print(f"Tokenizer vocab  : {len(tokenizer)}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model parameters : 300,768,256
Tokenizer vocab  : 384


## Cell 6 — Tokenize dataset

In [6]:
def tokenize(batch):
    enc = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )
    enc["labels"] = labels["input_ids"]
    return enc


tokenized_ds = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

# Small fixed subset for fast validation during training
eval_subset = tokenized_ds["val"].shuffle(seed=SEED).select(
    range(min(EVAL_SUBSET_SIZE, len(tokenized_ds["val"])))
)
print(f"eval_subset size : {len(eval_subset)}")

Tokenizing:   0%|          | 0/65120 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/8122 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/8122 [00:00<?, ? examples/s]

eval_subset size : 500


## Cell 7 — Metrics & callback

In [7]:
VOCAB_UPPER = len(tokenizer)


def safe_batch_decode(token_id_array):
    arr = np.asarray(token_id_array)
    bad = (arr < 0) | (arr >= VOCAB_UPPER)
    if bad.any():
        arr = np.where(bad, tokenizer.pad_token_id, arr)
    return tokenizer.batch_decode(arr, skip_special_tokens=True)


def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    pred_texts  = [normalize_target(x) for x in safe_batch_decode(predictions)]
    label_texts = [normalize_target(x) for x in safe_batch_decode(
        np.where(labels == -100, tokenizer.pad_token_id, labels)
    )]

    exact    = [int(p == g)                for p, g in zip(pred_texts, label_texts)]
    exact_ci = [int(p.lower() == g.lower()) for p, g in zip(pred_texts, label_texts)]
    lev      = [Levenshtein.ratio(p, g)    for p, g in zip(pred_texts, label_texts)]

    # Print a few examples every eval
    print("\nSample predictions:")
    for i in range(min(4, len(pred_texts))):
        flag = "✓" if pred_texts[i] == label_texts[i] else "✗"
        print(f"  [{flag}] PRED : {pred_texts[i]!r}")
        print(f"       GOLD : {label_texts[i]!r}")

    return {
        "exact_match":            float(np.mean(exact)),
        "exact_match_ci":         float(np.mean(exact_ci)),
        "levenshtein_similarity": float(np.mean(lev)),
    }


class MetricsTableCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics:
            return
        step     = int(state.global_step)
        val_loss = metrics.get("eval_loss",                   float("nan"))
        em       = metrics.get("eval_exact_match",            float("nan"))
        em_ci    = metrics.get("eval_exact_match_ci",         float("nan"))
        lev      = metrics.get("eval_levenshtein_similarity", float("nan"))
        print(
            f"\n{'─'*60}\n"
            f"  STEP {step:>6} │ loss={val_loss:.4f} │ "
            f"EM={em:.4f} ({em*100:.1f}%) │ EM_CI={em_ci:.4f} │ LEV={lev:.4f}"
            f"\n{'─'*60}"
        )

## Cell 8 — Training arguments

In [8]:
# ── Cell 8 — Training Arguments ───────────────────────────────────────────────
import os, math, torch
from pathlib import Path

# ✅ CRITICAL: must be under /kaggle/working/ to persist in Save & Run mode
OUTPUT_DIR = Path("/kaggle/working/output")
CKPT_DIR   = OUTPUT_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_DIR = Path("/kaggle/input/datasets/mennakhalifavv/checkpointmenna/output/checkpoints")
def find_latest_checkpoint(ckpt_dir: Path):
    if not ckpt_dir.exists():
        return None
    ckpts = sorted(
        [d for d in ckpt_dir.iterdir()
         if d.is_dir() and d.name.startswith("checkpoint-")],
        key=lambda x: int(x.name.split("-")[-1]),
    )
    return str(ckpts[-1]) if ckpts else None

train_size      = len(tokenized_ds["train"])
effective_batch = TRAIN_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count())
steps_per_epoch = max(1, math.ceil(train_size / effective_batch))
total_steps     = steps_per_epoch * NUM_TRAIN_EPOCHS
warmup_steps    = min(500, int(total_steps * 0.06))

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16

resume_ckpt = find_latest_checkpoint(DATASET_DIR)

print(f"effective_batch : {effective_batch}")
print(f"steps_per_epoch : {steps_per_epoch}")
print(f"total_steps     : {total_steps}")
print(f"warmup_steps    : {warmup_steps}")
print(f"use_bf16        : {use_bf16}  |  use_fp16 : {use_fp16}")
print(f"resume_ckpt     : {resume_ckpt}")
print(f"CKPT_DIR        : {CKPT_DIR}  (exists={CKPT_DIR.exists()})")

# ── For 15 epochs, increase patience & eval frequency accordingly ──────────────
NUM_TRAIN_EPOCHS         = 15
EARLY_STOPPING_PATIENCE  = 5   # allow more room before stopping in 15-epoch run

# Eval every ~1 epoch (so you get 15 checkpoints max, keeping best 5)
EVAL_STEPS = steps_per_epoch   

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CKPT_DIR),
    # ── Eval / save ──────────────────────────────────────────────────────────
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    logging_strategy="steps",
    logging_steps=50,
    save_total_limit=5,              # ✅ keep more checkpoints (was 3)
    load_best_model_at_end=True,
    metric_for_best_model="eval_exact_match",
    greater_is_better=True,
    # ── Batch / gradient ─────────────────────────────────────────────────────
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    # ── Epochs / LR ──────────────────────────────────────────────────────────
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    max_grad_norm=1.0,
    label_smoothing_factor=LABEL_SMOOTHING,
    # ── Generation (during eval) ─────────────────────────────────────────────
    optim="adafactor",
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    generation_num_beams=2,
    # ── Precision ────────────────────────────────────────────────────────────
    bf16=use_bf16,
    fp16=use_fp16,
    # ── Dataloader ───────────────────────────────────────────────────────────
    dataloader_num_workers=2,
    dataloader_pin_memory=False,     # ✅ False avoids the pin_memory warning on CPU-only
    report_to="none",
    seed=SEED,
    remove_unused_columns=True,
)

effective_batch : 64
steps_per_epoch : 1018
total_steps     : 10180
warmup_steps    : 500
use_bf16        : False  |  use_fp16 : True
resume_ckpt     : /kaggle/input/datasets/mennakhalifavv/checkpointmenna/output/checkpoints/checkpoint-7126
CKPT_DIR        : /kaggle/working/output/checkpoints  (exists=True)


## Cell 9 — Train

In [9]:
# ── Cell 9 — Train ────────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=eval_subset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
        MetricsTableCallback(),
    ],
)

train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
print(train_result)

# ✅ CRITICAL for Save & Run: explicitly save final model + tokenizer to /kaggle/working/
FINAL_MODEL_DIR = OUTPUT_DIR / "final_model"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

# ✅ Save training metrics too
import json
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

# ✅ Verify checkpoints actually exist
saved = list(CKPT_DIR.glob("checkpoint-*"))
print(f"\n✅ Checkpoints saved ({len(saved)}):")
for c in sorted(saved):
    print(f"   {c}")
print(f"\n✅ Final model saved to: {FINAL_MODEL_DIR}")

Step,Training Loss,Validation Loss,Exact Match,Exact Match Ci,Levenshtein Similarity
8144,7.530659,0.956651,0.724000,0.724000,0.989555
9162,7.510825,0.955954,0.748000,0.748000,0.990071
10180,7.470357,0.956409,0.752000,0.752000,0.990552
11198,7.458866,0.956699,0.756000,0.756000,0.990604
12216,7.450041,0.957270,0.758000,0.758000,0.990292
13234,7.441082,0.956563,0.764000,0.764000,0.990596
14252,7.436031,0.956939,0.758000,0.758000,0.990436
15270,7.430620,0.957010,0.762000,0.762000,0.990653


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Sample predictions:
  [✓] PRED : 'pr ḫrw n f m Wpi̯ rnpt m Ḏḥwtyt Tpj rnpt Wꜣg'
       GOLD : 'pr ḫrw n f m Wpi̯ rnpt m Ḏḥwtyt Tpj rnpt Wꜣg'
  [✓] PRED : 'zꜣt s jḥt'
       GOLD : 'zꜣt s jḥt'
  [✓] PRED : 'zꜣ f ḥtpy mꜣꜥ ḫrw'
       GOLD : 'zꜣ f ḥtpy mꜣꜥ ḫrw'
  [✓] PRED : 'sbjw k r nmt 〈f〉 nn wnn f'
       GOLD : 'sbjw k r nmt 〈f〉 nn wnn f'

────────────────────────────────────────────────────────────
  STEP   8144 │ loss=0.9567 │ EM=0.7240 (72.4%) │ EM_CI=0.7240 │ LEV=0.9896
────────────────────────────────────────────────────────────


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...


Sample predictions:
  [✓] PRED : 'pr ḫrw n f m Wpi̯ rnpt m Ḏḥwtyt Tpj rnpt Wꜣg'
       GOLD : 'pr ḫrw n f m Wpi̯ rnpt m Ḏḥwtyt Tpj rnpt Wꜣg'
  [✓] PRED : 'zꜣt s jḥt'
       GOLD : 'zꜣt s jḥt'
  [✓] PRED : 'zꜣ f ḥtpy mꜣꜥ ḫrw'
       GOLD : 'zꜣ f ḥtpy mꜣꜥ ḫrw'
  [✓] PRED : 'sbjw k r nmt 〈f〉 nn wnn f'
       GOLD : 'sbjw k r nmt 〈f〉 nn wnn f'

────────────────────────────────────────────────────────────
  STEP  15270 │ loss=0.9570 │ EM=0.7620 (76.2%) │ EM_CI=0.7620 │ LEV=0.9907
────────────────────────────────────────────────────────────


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15270, training_loss=3.9803816649831036, metrics={'train_runtime': 40230.54, 'train_samples_per_second': 24.28, 'train_steps_per_second': 0.38, 'total_flos': 2.0717596247565926e+17, 'train_loss': 3.9803816649831036, 'epoch': 15.0})


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** train metrics *****
  epoch                    =        15.0
  total_flos               = 192947650GF
  train_loss               =      3.9804
  train_runtime            = 11:10:30.54
  train_samples_per_second =       24.28
  train_steps_per_second   =        0.38

✅ Checkpoints saved (5):
   /kaggle/working/output/checkpoints/checkpoint-11198
   /kaggle/working/output/checkpoints/checkpoint-12216
   /kaggle/working/output/checkpoints/checkpoint-13234
   /kaggle/working/output/checkpoints/checkpoint-14252
   /kaggle/working/output/checkpoints/checkpoint-15270

✅ Final model saved to: /kaggle/working/output/final_model


## Cell 10 — Save best model

In [10]:
# Run this in a new cell anytime to check
from pathlib import Path
CKPT_DIR = Path("/kaggle/working/output/checkpoints")
ckpts = sorted(CKPT_DIR.glob("checkpoint-*"))
print(f"Found {len(ckpts)} checkpoints:")
for c in ckpts:
    size = sum(f.stat().st_size for f in c.rglob("*") if f.is_file())
    print(f"  {c.name}  →  {size/1e6:.1f} MB")

Found 5 checkpoints:
  checkpoint-11198  →  1205.1 MB
  checkpoint-12216  →  1205.2 MB
  checkpoint-13234  →  1205.2 MB
  checkpoint-14252  →  1205.2 MB
  checkpoint-15270  →  1205.2 MB


In [11]:
SAVE_DIR = OUTPUT_DIR / "best_model"
trainer.save_model(str(SAVE_DIR))
tokenizer.save_pretrained(str(SAVE_DIR))
print(f"Best model saved to: {SAVE_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved to: /kaggle/working/output/best_model


## Cell 11 — Free memory before final eval
**This is the crash-fix cell. Never skip it.**

In [12]:
# Release the trainer (holds references to optimizer, gradient state, etc.)
del trainer
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed ✓")
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f"Allocated: {allocated:.2f} GB  |  Reserved: {reserved:.2f} GB")

GPU memory freed ✓
Allocated: 1.23 GB  |  Reserved: 1.28 GB


## Cell 12 — Final evaluation with beam search

In [13]:
device    = "cuda" if torch.cuda.is_available() else "cpu"
gen_model = model.module if hasattr(model, "module") else model
gen_model = gen_model.to(device).eval()


@torch.no_grad()
def assemble_batch(texts, num_beams=FINAL_NUM_BEAMS, batch_size=4):
    """
    batch_size=4 is safe for beam=4 on P100/T4.
    Increase to 8 only if you have ≥16 GB VRAM and short sequences.
    """
    outputs = []
    for start in range(0, len(texts), batch_size):
        batch = [normalize_input(t) for t in texts[start : start + batch_size]]
        enc = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LEN,
        ).to(device)
        out = gen_model.generate(
            **enc,
            max_length=MAX_TARGET_LEN,
            num_beams=num_beams,
            early_stopping=True,
        )
        outputs.extend(
            [normalize_target(x) for x in tokenizer.batch_decode(out, skip_special_tokens=True)]
        )
        # Free intermediate tensors every batch
        del enc, out
        torch.cuda.empty_cache()
    return outputs


def evaluate_frame(frame, name, limit=FINAL_EVAL_LIMIT):
    if limit is not None and limit < len(frame):
        frame = frame.sample(limit, random_state=SEED).copy()
    preds = assemble_batch(frame["input_text"].tolist())
    golds = frame["target_text"].map(normalize_target).tolist()
    exact = float(np.mean([p == g for p, g in zip(preds, golds)]))
    lev   = float(np.mean([Levenshtein.ratio(p, g) for p, g in zip(preds, golds)]))
    print(f"\n{'='*50}")
    print(f"  {name}  (n={len(frame):,})")
    print(f"  Exact Match : {exact:.4f}  ({exact*100:.1f}%)")
    print(f"  Levenshtein : {lev:.4f}")
    print(f"{'='*50}")
    return preds, golds


val_preds,  val_golds  = evaluate_frame(val_df,  "Validation")
test_preds, test_golds = evaluate_frame(test_df, "Test")


  Validation  (n=1,000)
  Exact Match : 0.7430  (74.3%)
  Levenshtein : 0.9886

  Test  (n=1,000)
  Exact Match : 0.7010  (70.1%)
  Levenshtein : 0.9860


## Cell 13 — Quick inference examples

In [14]:
examples = [
    "n ḏ w d i ̯ r s",
    "n ṯ w ꞽ m s n",
    "ꜥ ḥ ꜥ",
    "ḥ t p ẖ n m w",
]

print("Quick inference examples:")
print("─" * 40)
for src, pred in zip(examples, assemble_batch(examples)):
    print(f"INPUT : {src}")
    print(f"PRED  : {pred}")
    print()

Quick inference examples:
────────────────────────────────────────
INPUT : n ḏ w d i ̯ r s
PRED  : nḏ wdi̯ r s

INPUT : n ṯ w ꞽ m s n
PRED  : n ṯw ꞽm sn

INPUT : ꜥ ḥ ꜥ
PRED  : ꜥḥꜥ

INPUT : ḥ t p ẖ n m w
PRED  : ḥtp ẖnmw



## Cell 14 — Save predictions to CSV (optional)

In [15]:
val_results = pd.DataFrame({
    "input_text":  val_df["input_text"].iloc[:len(val_preds)].values,
    "gold":        val_golds,
    "pred":        val_preds,
    "correct":     [int(p == g) for p, g in zip(val_preds, val_golds)],
    "lev":         [round(Levenshtein.ratio(p, g), 4) for p, g in zip(val_preds, val_golds)],
})
val_results.to_csv(OUTPUT_DIR / "val_predictions.csv", index=False)

test_results = pd.DataFrame({
    "input_text":  test_df["input_text"].iloc[:len(test_preds)].values,
    "gold":        test_golds,
    "pred":        test_preds,
    "correct":     [int(p == g) for p, g in zip(test_preds, test_golds)],
    "lev":         [round(Levenshtein.ratio(p, g), 4) for p, g in zip(test_preds, test_golds)],
})
test_results.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

print(f"Predictions saved to {OUTPUT_DIR}")
print(f"Val  errors (first 10):")
print(val_results[val_results["correct"] == 0].head(10)[["input_text","gold","pred","lev"]].to_string())

Predictions saved to /kaggle/working/output
Val  errors (first 10):
                                                                                                                         input_text                                                                gold                                                         pred     lev
5                                                                                                                   ꜥ k n k w s ꞽ r             Ppy Nfr kꜣ Rꜥw pw nb sḫm ḫrw jwtj rnw f ꜣꜣrw ḫt m ḥnktt     Ppy Nfr kꜣ Rꜥw pw nb sḫm ḫrw jwtj r nw fꜣ ꜣrw ḫt m ḥnktt  0.9730
15                                                                                                      d w ꜣ n j j m j w k k w s n                                                     Wsjr nb Jwnš〈t〉                                             Wsjr nb J wnš〈t〉  0.9677
22                                                                                                  j y n j r m ꜣ ꜣ n ṯ r j m